# Demo 02 — Privacy Sanitization

This notebook demonstrates the **PII sanitization** pipeline built into SynthoHive. It shows how to:

1. Generate sample data containing PII (names, emails, phone numbers, SSNs, credit cards)
2. Configure privacy rules (built-in defaults + custom rules)
3. Detect PII columns automatically with `PIISanitizer.analyze()`
4. Sanitize the data with `PIISanitizer.sanitize()`

No external data files or Spark are required.

In [ ]:
import numpy as np
import pandas as pd

from syntho_hive.privacy.sanitizer import PIISanitizer, PrivacyConfig, PiiRule

## 1. Generate Raw Data with PII

We create a 50-row table containing realistic PII fields: first/last names, emails, phone numbers, SSNs, credit card numbers, and custom loyalty IDs.

In [ ]:
def make_raw_users(num_rows: int) -> pd.DataFrame:
    rng = np.random.default_rng(7)
    first_names = ["Alex", "Jordan", "Sam", "Taylor", "Jamie", "Riley", "Casey", "Drew"]
    last_names = ["Lee", "Patel", "Garcia", "Chen", "Olsen", "Diaz", "Nguyen", "Brown"]
    cities = ["NY", "SF", "SEA", "DAL", "BOS"]

    rows = []
    for i in range(num_rows):
        first = rng.choice(first_names)
        last = rng.choice(last_names)
        city = rng.choice(cities)
        email = f"{first.lower()}.{last.lower()}{i}@example.com"
        phone = f"({rng.integers(200, 999)})-{rng.integers(200, 999)}-{rng.integers(1000, 9999)}"
        ssn = f"{rng.integers(100, 999):03d}-{rng.integers(10, 99):02d}-{rng.integers(1000, 9999):04d}"
        credit_card = f"{rng.integers(1000, 9999):04d}-{rng.integers(1000, 9999):04d}-{rng.integers(1000, 9999):04d}-{rng.integers(1000, 9999):04d}"
        loyalty_id = f"L-{rng.integers(10_000, 99_999)}"

        rows.append(
            {
                "user_id": i + 1,
                "first_name": first,
                "last_name": last,
                "city": city,
                "email": email,
                "phone": phone,
                "ssn": ssn,
                "credit_card": credit_card,
                "loyalty_id": loyalty_id,
                "notes": f"Called support on ticket {rng.integers(1000, 9999)}",
            }
        )

    return pd.DataFrame(rows)

raw_df = make_raw_users(50)
print(f"Raw data shape: {raw_df.shape}")
raw_df.head()

## 2. Configure Privacy Rules

`PrivacyConfig.default()` provides built-in rules for common PII patterns (email, phone, SSN, credit card, etc.). We extend it with a custom rule to **hash** loyalty IDs matching the pattern `L-\d{5}`.

In [ ]:
config = PrivacyConfig.default()
config.rules.append(
    PiiRule(
        name="loyalty_id",
        patterns=[r"L-\d{5}"],
        action="hash",
    )
)

print(f"Total PII rules: {len(config.rules)}")
for rule in config.rules:
    print(f"  - {rule.name}: action={rule.action}")

## 3. Detect PII Columns

The `analyze()` method scans each column in the DataFrame and returns a mapping of column names to the PII rule that matched.

In [ ]:
sanitizer = PIISanitizer(config=config)
detected = sanitizer.analyze(raw_df)

print("Detected PII columns:")
for col, rule_name in detected.items():
    print(f"  {col}: {rule_name}")

## 4. Sanitize the Data

The `sanitize()` method applies the configured action (mask, fake, hash, etc.) to each detected PII column.

In [ ]:
sanitized_df = sanitizer.sanitize(raw_df, pii_map=detected)

print(f"Sanitized data shape: {sanitized_df.shape}")
sanitized_df.head()

## 5. Compare Before and After

A side-by-side comparison of a few rows showing the raw and sanitized values.

In [ ]:
compare_cols = [c for c in detected.keys() if c in raw_df.columns]
print("=== Raw ===")
print(raw_df[compare_cols].head())
print("\n=== Sanitized ===")
print(sanitized_df[compare_cols].head())